# 🧙‍♂️ Counsellor in Action — Real GitHub Integration

Connecting Dots... 👉 https://sriharshacr.github.io/blogs/

## What This Notebook Shows

The production step-up from [counsellor_poc.ipynb](https://colab.research.google.com/github/SriharshaCR/blogs/blob/main/assets/notebooks/decision-archaeology-kba/counsellor_poc.ipynb).

The story is identical — memory hits 91%, Counsellor fires, the sniffer analyses, cleanup runs. What changes:

| POC | In Action |
|---|---|
| Laws hardcoded in the notebook | Laws fetched live from your GitHub `/laws/` folder |
| GitHub Issue simulated (print) | GitHub Issue created for real |
| One dependency (`openai`) | Two dependencies (`openai`, `PyGithub`) |

**Two keys required:**
- LLM key (Groq or HuggingFace — free, no credit card) — [set up in under 5 minutes](https://sriharshacr.github.io/blogs/guides/llm-free-apis/#groq-setup)
- GitHub fine-grained PAT (repo-scoped, Issues: Write only) — [minimal setup here](https://sriharshacr.github.io/blogs/guides/github-pat-setup/)

## GitHub Repository Setup

Create a private GitHub repo with a `/laws/` folder before running this notebook.

**Repo structure:**
```
your-username/counsellor-kba/
└── laws/
    └── law-of-memory-management.yaml
```

**Content of `laws/law-of-memory-management.yaml`:**
```yaml
name: Law of Memory Management
description: Prevents memory exhaustion by triggering cleanup when usage exceeds threshold
trigger: memory_pct > 0.90
trigger_threshold: 0.90
agent: agent-memory-sniffer
auto_rule: age_days > 15 AND relevance_days < 7
auto_rule_age_days: 15
auto_rule_relevance_days: 7
target: 0.80
by: Counsellor
```

Then generate a fine-grained PAT scoped to this repo with **Contents: Read** and **Issues: Write** permissions.
Full walkthrough: [GitHub PAT setup guide](https://sriharshacr.github.io/blogs/guides/github-pat-setup/)

### 📄 Copy-pasteable `laws/law-of-memory-management.yaml` Syntax

Create a file named `law-of-memory-management.yaml` in your `/laws/` directory and paste this exact content:

```yaml
name: Law of Memory Management
description: Prevents memory exhaustion by triggering cleanup when usage exceeds threshold
trigger: memory_pct > 0.90
trigger_threshold: 0.90
agent: agent-memory-sniffer
auto_rule: age_days > 15 AND relevance_days < 7
auto_rule_age_days: 15
auto_rule_relevance_days: 7
target: 0.80
by: Counsellor
```

## How to Interact With This Notebook

Run cells top to bottom. Each builds on the previous.

**Three places to configure:**
- **LLM config cell** — set `PROVIDER` to `groq` or `huggingface`
- **GitHub config cell** — set `REPO_NAME` to your repo (e.g. `your-username/counsellor-kba`)
- **Approval cell** — set `approved = True` after reviewing the GitHub Issue

**What you will see:**
1. Laws loaded live from your GitHub repo
2. Memory state before Counsellor runs
3. Which Law triggers and why
4. What the AI agent recommends (one LLM call)
5. A real GitHub Issue created in your repo (with a link)
6. Memory state after cleanup, Issue closed

In [ ]:
!pip install openai PyGithub -q
import json, base64
from google.colab import userdata
print('✅ Ready')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
# 🔌 SWAP: Change PROVIDER to switch between Groq and HuggingFace.
#   Full setup guide: https://sriharshacr.github.io/blogs/guides/llm-free-apis/

from google.colab import userdata
from openai import OpenAI

PROVIDER = 'groq'   # 'groq' | 'huggingface'

_CONFIGS = {
    'groq': {
        'base_url': 'https://api.groq.com/openai/v1',
        'secret':   'GROQ_API_KEY', # ⚠️ Expecting there is this Secret defined
        'model':    'openai/gpt-oss-safeguard-20b',   # JSON fallback should support
    },
    'huggingface': {
        'base_url': 'https://router.huggingface.co/v1',
        'secret':   'HF_TOKEN', # ⚠️ Expecting there is this Secret defined
        'model':    'openai/gpt-oss-120b:fastest',
    },
}

_cfg      = _CONFIGS[PROVIDER]
client    = OpenAI(base_url=_cfg['base_url'], api_key=userdata.get(_cfg['secret']))
LLM_MODEL = _cfg['model']

MEMORY_LIMIT      = 2200
TRIGGER_THRESHOLD = 0.90
HARD_CAP          = 0.95
TARGET_USAGE      = 0.80

print(f'Config loaded — {PROVIDER} / {LLM_MODEL}')
print(f'  Limit={MEMORY_LIMIT}c | Trigger>{TRIGGER_THRESHOLD:.0%} | Hard cap>{HARD_CAP:.0%} | Target<{TARGET_USAGE:.0%}')

In [ ]:
# ── GitHub Config ──────────────────────────────────────────────────────────
# 🔑 Add GITHUB_TOKEN to Colab Secrets before running.
#    Fine-grained PAT — Contents: Read, Issues: Write only.
#    Setup guide: https://sriharshacr.github.io/blogs/guides/github-pat-setup/

from github import Github

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') # ⚠️ Expecting there is this Secret defined
REPO_NAME    = 'SriharshaCR/test-playground'   # ← ⚠️change to your repo

g = Github(GITHUB_TOKEN)
try:
    repo       = g.get_repo(REPO_NAME)
    visibility = 'private' if repo.private else 'public'
    print(f'✅ Connected to: {repo.full_name}  ({visibility})')
except Exception as e:
    print(f'❌ Could not connect to {REPO_NAME}: {e}')
    print('   Check REPO_NAME and confirm GITHUB_TOKEN has Contents:Read permission.')
    print('   Setup guide: https://sriharshacr.github.io/blogs/guides/github-pat-setup/')
    raise

In [ ]:
# ── Fetch Laws from GitHub /laws/ ──────────────────────────────────────────
import yaml

print('📜 Fetching Laws from GitHub...')
print()
try:
    law_files = repo.get_contents('laws')
except Exception as e:
    print(f'❌ No /laws/ folder found in {REPO_NAME}: {e}')
    print('   Create a /laws/ folder with .yaml law files.')
    print('   See the GitHub Repo Setup cell above for the exact file format.')
    raise

LAWS = {}
for f in sorted(law_files, key=lambda x: x.name):
    if not f.name.endswith('.yaml'):
        continue
    raw       = base64.b64decode(f.content).decode('utf-8')
    law       = yaml.safe_load(raw)

    # Check if the YAML successfully loaded as a dictionary
    if not isinstance(law, dict):
        print(f'  ⚠️  Skipping {f.name}: File content is not a valid YAML dictionary.')
        continue

    key       = f.name.replace('.yaml', '').replace('-', '_')
    LAWS[key] = law
    name      = law.get('name', key)
    trigger   = law.get('trigger', '')
    agent     = law.get('agent', '')
    auto_r    = law.get('auto_rule', '')
    print(f'  ✅  {name}')
    print(f'       trigger:   {trigger}')
    print(f'       agent:     {agent}')
    print(f'       auto_rule: {auto_r}')

print()
print(f'Loaded {len(LAWS)} Law(s) from {REPO_NAME}/laws/')


## Understanding Memory Entries

Before looking at the entries, here is what each field means — because the agent's entire decision about what to keep, flag, or remove is based on reading these fields.

**`content`** — The actual knowledge stored. This is what gets retrieved when Counsellor or another agent needs context before making a decision. It can be a learned lesson, a behavioral constraint, a process step, or a personal preference. The content should be self-contained — it needs to make sense without any surrounding context, because it may be read months later by an agent that has no memory of how it was created.

**`category`** — The type of knowledge. This tells the agent how aggressively it can clean the entry:
| Category | What it stores | Auto-remove eligible? |
|---|---|---|
| `strategic` | Timeless principles — architectural decisions, design philosophy | Never |
| `lesson` | Learned from an incident or mistake | Never |
| `profile` | User preferences, locale, working style, system pointers | Never |
| `operational` | Process instructions — checklists, protocols, workflows | Only if stale by threshold |
| `tactical` | Short-term, time-bound facts — deadlines, blockers, reminders | Yes, aggressively |

**`relevance_days`** — How long this entry is expected to remain useful. A tactical reminder about a meeting next week might be relevant for 3 days. A strategic architectural principle is relevant for 365. This is not set automatically — it is declared when the entry is created, as a commitment about the entry's expected useful life.

**`age_days`** — How many days ago this entry was written. The agent combines this with `relevance_days` to detect entries that have outlived their usefulness: if an entry is older than it was ever meant to be relevant, it is a cleanup candidate.

**The auto-remove rule** is simple: `age_days > 15 AND relevance_days < 7`. An old tactical entry — past its expected useful life — gets removed automatically. No human input needed. Anything that does not clearly match this rule gets flagged for your decision instead.

In [ ]:
# ── Simulated Memory State ─────────────────────────────────────────────────
# Realistic placeholder entries. Swap with your own MEMORY.md content.
# Fields: key | content | category | relevance_days | age_days

MEMORY = [
    # ─ Strategic (permanent) ─────────────────────────────────────────
    {'key': 's1', 'content': 'Architectural Protocol: prefer composition over inheritance. Validate at system boundaries, trust internals. Complexity is a cost that must be justified.', 'category': 'strategic', 'relevance_days': 365, 'age_days': 45},
    {'key': 's2', 'content': 'Simplicity Definition: a solution is simple when removing any part breaks it. Never add abstractions ahead of need.', 'category': 'strategic', 'relevance_days': 365, 'age_days': 30},
    # ─ Lessons (long-term) ───────────────────────────────────────────
    {'key': 'l1', 'content': 'Never update shared config files without notifying all dependent services. An email service outage was traced to an unannounced config change.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 20},
    {'key': 'l2', 'content': 'Always version-lock external dependencies in agent workflows. Upstream changes broke the summariser skill twice in one month.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 18},
    {'key': 'l3', 'content': 'Agents that write to disk need explicit TTL policies. Without them, temp files accumulate silently and cause disk pressure over weeks.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 12},
    {'key': 'l4', 'content': 'Never deploy on Fridays or before public holidays. Two incidents were traced to weekend deployments with no on-call coverage.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 8},
    {'key': 'l5', 'content': 'Service-to-service calls must include correlation IDs for distributed tracing. Debugging a cross-service incident took 4 hours due to missing trace context.', 'category': 'lesson', 'relevance_days': 180, 'age_days': 6},
    # ─ Profile (permanent) ───────────────────────────────────────────
    {'key': 'p1', 'content': 'User prefers direct communication, no padding. Values evidence over opinion. Batch questions, no rapid-fire. Summarise decisions at end of session.', 'category': 'profile', 'relevance_days': 365, 'age_days': 60},
    {'key': 'p2', 'content': 'Locale: IST (UTC+5:30). Working hours 9AM-7PM IST. Peak focus 10AM-1PM and 8PM-11PM. Async-first outside those windows.', 'category': 'profile', 'relevance_days': 365, 'age_days': 60},
    {'key': 'p3', 'content': 'KBA System pointer: github.com/counsellor-demo/kba - Laws, Decisions, Patterns in GitHub Discussions. Memory stores URL pointer only.', 'category': 'profile', 'relevance_days': 365, 'age_days': 5},
    # ─ Operational (medium-term) ──────────────────────────────────────
    {'key': 'o1', 'content': 'Deployment checklist: run tests locally, push to staging, validate metrics, get approval, deploy prod, monitor for 30 mins.', 'category': 'operational', 'relevance_days': 90, 'age_days': 25},
    {'key': 'o2', 'content': 'Code review: read PR description, check tests, review logic, check edge cases, comment with reasoning, approve or request changes.', 'category': 'operational', 'relevance_days': 90, 'age_days': 30},
    {'key': 'o3', 'content': 'Meeting protocol: set agenda 24h before, timeboxed items, log decisions, assign action items with owners and dates, follow up within 48h.', 'category': 'operational', 'relevance_days': 90, 'age_days': 22},
    {'key': 'o4', 'content': 'Incident response: open GitHub issue with incident label, assign on-call, post status every 30 mins, conduct postmortem within 48 hours.', 'category': 'operational', 'relevance_days': 90, 'age_days': 15},
    # ─ Tactical (STALE — auto-remove candidates) ──────────────────────
    {'key': 't1', 'content': 'Active trial: shared services POC deadline was Oct 15. Review meeting Oct 17. Outcome: approved for phase 2.', 'category': 'tactical', 'relevance_days': 3, 'age_days': 22},
    {'key': 't2', 'content': 'Blocker Oct 20: waiting for infra team to provision staging namespace. ETA 2 days. Follow up if not resolved by Oct 22.', 'category': 'tactical', 'relevance_days': 2, 'age_days': 18},
    {'key': 't3', 'content': 'Reminder Oct 28: weekly memory review. Agenda - check trial progress, archive completed items, update KBA pointers.', 'category': 'tactical', 'relevance_days': 1, 'age_days': 16},
]

total = sum(len(e['content']) for e in MEMORY)
pct   = total / MEMORY_LIMIT
print(f'Memory loaded: {len(MEMORY)} entries | {total}/{MEMORY_LIMIT} chars ({pct:.1%})')
print()
from collections import Counter
cats = Counter(e['category'] for e in MEMORY)
for cat, n in sorted(cats.items()):
    cat_chars = sum(len(e['content']) for e in MEMORY if e['category'] == cat)
    print(f'  {cat:<12} {n} entries  {cat_chars} chars')

In [ ]:
# ── BEFORE: No Law Active ──────────────────────────────────────────────────
if pct >= HARD_CAP:
    status = '🔴 CRITICAL'
elif pct >= TRIGGER_THRESHOLD:
    status = '🟡 WARNING'
else:
    status = '🟢 OK'

print('=' * 55)
print('  BEFORE — No Counsellor Law active')
print('=' * 55)
print(f'  Memory:  {total}/{MEMORY_LIMIT} chars  ({pct:.1%})  {status}')
print()
print('  Without Counsellor:')
print('  ✗  Memory keeps filling silently')
print(f'  ✗  At {HARD_CAP:.0%}: new memories rejected')
print('  ✗  Learnings from this session may not persist')
print('  ✗  Cleanup only happens when someone notices')

In [ ]:
# ── Counsellor: Check Laws (fetched from GitHub) ───────────────────────────
print('🧙‍♂️ Counsellor scanning Laws...')
print()

triggered = None
for law_id, law in LAWS.items():
    threshold = float(law.get('trigger_threshold', 1.0))
    trigger   = law.get('trigger', '')
    if 'memory_pct' in trigger and pct > threshold:
        triggered = law
        law_name  = law.get('name', law_id)
        law_agent = law.get('agent', '')
        print(f'  🔴 {law_name} triggered')
        print(f'     Condition: {pct:.1%} > {threshold:.0%}')
        print(f'     Activating: `{law_agent}`')
        break

if not triggered:
    print('  ✅ All Laws satisfied. No action required.')

In [ ]:
# ── agent-memory-sniffer: One LLM Call ─────────────────────────────────────
if triggered is None:
    print('⚠️ Cannot run analysis: No active Law was triggered.')
    print('   Please ensure a valid law configuration file (like law-of-memory-management.yaml)')
    print('   is loaded in your GitHub repository and satisfies the trigger threshold.')
else:
    law_name       = triggered.get('name', '')
    law_agent      = triggered.get('agent', 'agent-memory-sniffer')
    auto_rule      = triggered.get('auto_rule', 'age_days > 15 AND relevance_days < 7')
    trigger_thresh = float(triggered.get('trigger_threshold', TRIGGER_THRESHOLD))
    cleanup_target = float(triggered.get('target', TARGET_USAGE))

    memory_lines = []
    for e in MEMORY:
        k   = e['key']
        cat = e['category']
        age = e['age_days']
        rel = e['relevance_days']
        txt = e['content'][:80]
        memory_lines.append(f'  [{k}] {cat} | age={age}d | relevance={rel}d')
        memory_lines.append(f'  content: {txt}...')
    memory_summary = '\n'.join(memory_lines)

    prompt_lines = [
        f'You are {law_agent}, activated by Counsellor under the {law_name}.',
        '',
        'CURRENT STATE',
        f'Usage: {total}/{MEMORY_LIMIT} chars ({pct:.1%})',
        f'Trigger: >{trigger_thresh:.0%} | Hard cap: >{HARD_CAP:.0%} | Target: <{cleanup_target:.0%}',
        '',
        'AUTO-REMOVE RULE (no human input needed)',
        f'Remove entries where: {auto_rule}',
        'NEVER auto-remove categories: strategic, profile, lesson',
        '',
        'MEMORY ENTRIES',
        memory_summary,
        '',
        'Return valid JSON with exactly these keys:',
        '- title (string): short issue title',
        '- problem (string): one sentence describing the problem',
        '- impact (list of strings): 2-3 impact items',
        '- root_cause (string): one sentence',
        '- auto_remove (list of strings): entry keys matching the auto-remove rule only',
        '- flag_for_human (list of strings): entry keys needing human decision, format: key | reason',
        '- keep (list of strings): entry keys that are safe and should stay',
        '- est_savings_chars (integer): estimated chars freed by auto-remove',
    ]
    prompt = '\n'.join(prompt_lines)

    print('🔍 agent-memory-sniffer analysing entries...')
    print()

    resp   = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1,
        response_format={'type': 'json_object'},
    )
    report = json.loads(resp.choices[0].message.content)

    auto_keys   = report.get('auto_remove', [])
    flag_items  = report.get('flag_for_human', [])
    flag_keys   = [f.split('|')[0].strip() for f in flag_items]
    keep_count  = len(report.get('keep', []))
    est_savings = report.get('est_savings_chars', 0)

    print('✅ Analysis complete')
    print(f'  Auto-remove: {auto_keys}')
    print(f'  Flag:        {flag_keys}')
    print(f'  Keep:        {keep_count} entries')
    print(f'  Est. savings: ~{est_savings} chars')


In [ ]:
# ── Create Real GitHub Issue ────────────────────────────────────────────────
law_name      = triggered.get('name', '')
law_agent     = triggered.get('agent', '')
auto_rule     = triggered.get('auto_rule', '')
auto_remove   = report.get('auto_remove', [])
flag_for_human = report.get('flag_for_human', [])
est_saved     = sum(len(e['content']) for e in MEMORY if e['key'] in auto_remove)

lines = [
    f'**Flagged by:** `{law_agent}`',
    f'**Law:** {law_name}',
    '',
    '---',
    '### Problem',
    report.get('problem', ''),
    '',
    '### Impact',
]
lines += ['- ' + i for i in report.get('impact', [])]
lines += [
    '',
    '### Root Cause',
    report.get('root_cause', ''),
    '',
    '---',
    '### Proposed Actions',
    '',
    f'**✅ Auto-remove** ({auto_rule}):',
]
lines += ['- `' + k + '`' for k in auto_remove]
lines += [
    f'*Estimated savings: ~{est_saved} chars*',
    '',
    '**⚠️ Flag for your review:**',
]
lines += ['- ' + f for f in flag_for_human]
lines += [
    '',
    '---',
    '*Maintained by Counsellor 🧙‍♂️ · [Decision Archaeology post](https://sriharshacr.github.io/blogs/decision-archaeology-kba/)*',
]
issue_body = '\n'.join(lines)

# Create labels silently if they do not already exist
for label_name, color in [('memory-management', 'e4e669'), ('counsellor-auto', '0075ca')]:
    try:
        repo.create_label(label_name, color)
    except Exception:
        pass  # label already exists

issue = repo.create_issue(
    title  = report.get('title', 'Memory cleanup required'),
    body   = issue_body,
    labels = ['memory-management', 'counsellor-auto'],
)
ISSUE_URL = issue.html_url
print(f'✅ Issue #{issue.number} created in {REPO_NAME}')
print(f'   {ISSUE_URL}')

## Review the Issue, Then Approve

Open the issue URL printed above. Review:
- **Auto-remove** entries: proposed based on the Law's rule (`age_days > 15 AND relevance_days < 7`)
- **Flagged** entries: the agent is uncertain — your decision

When ready:
- Set `approved = True` in the next cell to run cleanup
- Set `approved = False` to hold without changes

In [ ]:
# ── Your Approval ──────────────────────────────────────────────────────────
print(f'📋 Review at: {ISSUE_URL}')
print()

approved = True   # ← set False to hold

if approved:
    issue.create_comment('✅ Approved by owner. Running auto-cleanup.')
    print('✅ Approved — running auto-cleanup')
else:
    print(f'⏸  Pending — review at: {ISSUE_URL}')
    print('   Set approved = True to proceed')

In [ ]:
# ── Auto-cleanup ────────────────────────────────────────────────────────────
if approved:
    auto_remove    = report.get('auto_remove', [])
    flag_for_human = report.get('flag_for_human', [])

    remove_keys = set(auto_remove)
    removed     = [e for e in MEMORY if e['key'] in remove_keys]
    cleaned     = [e for e in MEMORY if e['key'] not in remove_keys]
    freed       = sum(len(e['content']) for e in removed)

    print(f'Removed {len(removed)} stale entries ({freed} chars freed):')
    for e in removed:
        k   = e['key']
        age = e['age_days']
        rel = e['relevance_days']
        txt = e['content'][:55]
        print(f'  ❌ [{k}] age={age}d | relevance={rel}d | {txt}...')

    if flag_for_human:
        print(f'\n⚠️  {len(flag_for_human)} entries await your decision:')
        for item in flag_for_human:
            parts  = item.split('|')
            k      = parts[0].strip()
            reason = parts[1].strip() if len(parts) > 1 else ''
            entry  = next((e for e in MEMORY if e['key'] == k), None)
            if entry:
                txt = entry['content'][:55]
                print(f'  ? [{k}] {reason} | {txt}...')
else:
    cleaned = MEMORY
    freed   = 0
    print('No changes applied.')

In [ ]:
# ── AFTER ──────────────────────────────────────────────────────────────────
new_total      = sum(len(e['content']) for e in cleaned)
new_pct        = new_total / MEMORY_LIMIT
cleanup_target = float(triggered.get('target', TARGET_USAGE))
target_ok      = new_pct < cleanup_target
target_msg     = '✅ Met' if target_ok else '⚠️  Review flagged entries'
law_name       = triggered.get('name', '')

print('=' * 55)
print('  AFTER — Counsellor Law executed')
print('=' * 55)
print('       ' + 'BEFORE'.rjust(22) + '  ' + 'AFTER'.rjust(22))
print('       ' + ('-' * 22) + '  ' + ('-' * 22))
print(f'  Usage: {total:>5}/{MEMORY_LIMIT} ({pct:.1%})   →   {new_total:>5}/{MEMORY_LIMIT} ({new_pct:.1%})')
print(f'  Entries: {len(MEMORY):>19}   →   {len(cleaned):>19}')
print()
print(f'  Target (<{cleanup_target:.0%}):  {target_msg}')
print(f'  Chars freed:   {freed}')
print()
print('  What just happened:')
print(f'  1. Memory hit {pct:.1%}  →  {law_name} triggered (fetched from GitHub)')
print( '  2. agent-memory-sniffer analysed entries, produced a plan (1 LLM call)')
print(f'  3. GitHub Issue #{issue.number} created: {ISSUE_URL}')
print( '  4. Auto-remove: stale tactical entries deleted')
print( '  5. Flagged entries surfaced for your decision')
print(f'  6. Memory: {pct:.1%} → {new_pct:.1%}  (freed {freed} chars)')

if approved:
    issue.edit(state='closed')
    print(f'\n✅ Issue #{issue.number} closed after cleanup')
    print()
print('  At the core: GitHub project management.')
print('  Counsellor makes it special with the magical wand of AI. 🧙‍♂️')

---

*Read the full post: [Decision Archaeology — Why Decisions Die and How a KBA Fixes That](https://sriharshacr.github.io/blogs/decision-archaeology-kba/)*